In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [13]:
import re
train= pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(train.isnull().sum().sum())  #no nulls values

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\-\./\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["cleaned_prompt"] = train["prompt"].apply(clean_text)
for opt in ["A","B","C","D","E"]:
    train[f"cleaned_{opt}"] = train[f"{opt}"].apply(clean_text)
train.head(5)

0


,id,prompt,A,B,C,D,E,answer,cleaned_prompt,cleaned_A,cleaned_B,cleaned_C,cleaned_D,cleaned_E
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


In [20]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")
train["prompt_token"] = train["cleaned_prompt"].apply(nltk.word_tokenize)
for opt in ["A","B","C","D","E"]:
    train[f"{opt}_token"] = train[f"cleaned_{opt}"].apply(nltk.word_tokenize)

text = train["cleaned_prompt"].tolist()
for opt in ["A","B","C","D","E"]:
    text += train[f"cleaned_{opt}"].tolist()

tfidf = TfidfVectorizer()
tfidf.fit(text)

i = 0
p_vec = tfidf.transform([train.loc[i,"cleaned_prompt"]])
o_vec = [tfidf.transform([train.loc[i,f"cleaned_{opt}"]]) for opt in ["A","B","C","D","E"]]

cos_sim = [cosine_similarity(p_vec, opt)[0][0] for opt in o_vec]
cos_sim

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[np.float64(0.16679528994578402),
 np.float64(0.18754309767044808),
 np.float64(0.4200856621712537),
 np.float64(0.37036798616566935),
 np.float64(0.10824678045170937)]

In [ ]:
sample= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sample.to_csv('submission.csv', index=False)